# 00 - Train and compare ACGAN vs DCGAN

This notebook is **repo-script-first**: it tries to use the repository's original scripts and modules instead of reimplementing training logic inside the notebook.

## Repo usage / self-coded inventory

**Repo-original code used**

- `src.models.dcgan.Generator` and `src.models.dcgan.Discriminator`
- `src.models.acgan.Generator` and `src.models.acgan.Discriminator`
- Repo data folder convention: `data/processed/labelled_4232`

**Still self-coded in this notebook**

- DataLoader creation for GAN training
- DCGAN and ACGAN training loops
- Saving generated synthetic images into Google Drive
- IS/FID style comparison helper code

**Why still self-coded**

The repo has GAN model classes, but no original runnable `scripts/train_dcgan.py`, `scripts/train_acgan.py`, or `scripts/compare_gans.py`. So this notebook uses the repo architectures, but has to orchestrate GAN training itself.

In [ ]:
# ============================================================
# Colab setup: clone repo, mount Drive, install dependencies
# ============================================================
import os, sys, subprocess, json, shutil, textwrap, time
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print('Not running in Colab; continuing with local paths.')

REPO_URL = 'https://github.com/tlinhevg05/contrastive-synthesis-medcls_CVProject.git'
REPO_DIR = Path('/content/contrastive-synthesis-medcls_CVProject') if IN_COLAB else Path.cwd()

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.check_call(['git', 'clone', REPO_URL, str(REPO_DIR)])
    else:
        print('Repo already exists:', REPO_DIR)
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=False)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('Repo dir:', REPO_DIR)

# Install dependencies.
req = REPO_DIR / 'requirements.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=False)
if req.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)], check=False)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchvision', 'torchaudio', 'timm', 'PyYAML', 'scikit-learn',
    'seaborn', 'matplotlib', 'pandas', 'numpy', 'pillow', 'opencv-python', 'tqdm', 'wandb'
], check=False)

import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU, running on CPU')

# Disable W&B 
os.environ['WANDB_MODE'] = 'disabled'

# Paths
DRIVE_ROOT = Path('/content/drive/MyDrive/medcls_cvproject') if IN_COLAB else (REPO_DIR / 'local_drive_outputs')
DATA_ROOT = REPO_DIR / 'data' / 'processed'
LABELLED_DATA = DATA_ROOT / 'labelled_4232'
UNLABELLED_REAL_DATA = DATA_ROOT / 'unlabelled_16934' / 'images'
SYNTHETIC_DCGAN_DATA = DRIVE_ROOT / 'data' / 'processed' / 'synthetic_dcgan'
SYNTHETIC_ACGAN_DATA = DRIVE_ROOT / 'data' / 'processed' / 'synthetic_acgan'
OUTPUT_ROOT = DRIVE_ROOT / 'outputs_script_first'
CONFIG_OUT = REPO_DIR / 'configs' / 'generated_script_first'
CONFIG_OUT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('LABELLED_DATA:', LABELLED_DATA)
print('UNLABELLED_REAL_DATA:', UNLABELLED_REAL_DATA)
print('SYNTHETIC_DCGAN_DATA:', SYNTHETIC_DCGAN_DATA)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

# Change to 'smoke' for quick testing.
RUN_MODE = 'full'
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

In [ ]:
# ============================================================
# Helpers: write YAML and run repo scripts with logs
# ============================================================
import subprocess, sys, yaml, os, json, time
from pathlib import Path

def write_yaml(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        yaml.safe_dump(data, f, sort_keys=False)
    print('Wrote config:', path)
    print(yaml.safe_dump(data, sort_keys=False))
    return path

def run_and_log(cmd, log_path=None):
    cmd = [str(x) for x in cmd]
    print('Running command:')
    print(' '.join(cmd))
    if log_path is None:
        subprocess.run(cmd, check=True)
        return
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('w', encoding='utf-8') as f:
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            f.write(line)
        ret = process.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, cmd)

def count_images(root):
    root = Path(root)
    if not root.exists():
        return 0
    exts = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
    return sum(1 for p in root.rglob('*') if p.suffix.lower() in exts)

def find_checkpoint(output_dir):
    output_dir = Path(output_dir)
    candidates = [
        output_dir / 'best_model.pth',
        output_dir / 'best_finetune_model.pth',
        output_dir / 'checkpoint.pth',
        output_dir / 'model.pth',
        output_dir / 'classifier.pth',
    ]
    for p in candidates:
        if p.exists():
            return p
    all_ckpts = sorted(list(output_dir.rglob('*.pth')) + list(output_dir.rglob('*.pt')), key=lambda p: p.stat().st_mtime, reverse=True)
    return all_ckpts[0] if all_ckpts else None

In [ ]:
# ============================================================
# Notebook 00: Train 1 ACGAN + 4 class-specific DCGANs, save synthetic data to Drive
#   - 256x256 images for repo DCGAN discriminator
#   - greyscale X-rays repeated to 3 channels for repo models
#   - generated images saved as greyscale
#   - discriminator updated every 3 batches
#   - label smoothing
#   - lower discriminator LR
#   - small decaying instance noise
#   - preview grids saved during training
# ============================================================
from pathlib import Path
import os, json, math, random, shutil
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import save_image, make_grid

from src.models.dcgan import Generator as DCGANGenerator, Discriminator as DCGANDiscriminator
from src.models.acgan import Generator as ACGANGenerator, Discriminator as ACGANDiscriminator

assert LABELLED_DATA.exists(), f"Missing labelled data: {LABELLED_DATA}"
print('Labelled image count:', count_images(LABELLED_DATA))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if RUN_MODE == 'smoke':
    GAN_EPOCHS = 2
    MAX_IMAGES_PER_CLASS = 64
    N_SYNTH_PER_CLASS = 20
    PREVIEW_EVERY = 1
else:
    GAN_EPOCHS = 200
    MAX_IMAGES_PER_CLASS = None
    N_SYNTH_PER_CLASS = 1000
    PREVIEW_EVERY = 10

BATCH_SIZE = 32
NOISE_DIM = 100
ACGAN_NOISE_DIM = 128
IMG_SIZE = 256  # required by the repo DCGAN discriminator

# Report LR is 2e-5. Keep generator at report LR, make discriminator slightly weaker.
LR_G = 2e-5
LR_D = 1e-5


D_TRAIN_EVERY = 3

# Label smoothing. Helps avoid an overconfident discriminator.
REAL_LABEL = 0.9
FAKE_LABEL = 0.1

# Small noise added only to discriminator inputs, decays to 0.
INSTANCE_NOISE_START = 0.03

NUM_PREVIEW = 16

GAN_OUT = OUTPUT_ROOT / '00_gan_compare'
PREVIEW_DIR = GAN_OUT / 'previews'
SYNTHETIC_DCGAN_DATA.mkdir(parents=True, exist_ok=True)
SYNTHETIC_ACGAN_DATA.mkdir(parents=True, exist_ok=True)
GAN_OUT.mkdir(parents=True, exist_ok=True)
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

print('GAN output:', GAN_OUT)
print('Preview output:', PREVIEW_DIR)
print('Synthetic DCGAN save dir:', SYNTHETIC_DCGAN_DATA)
print('Synthetic ACGAN save dir:', SYNTHETIC_ACGAN_DATA)

# Clear old synthetic images from previous bad runs to avoid mixing outputs.
# Comment these two lines out if you intentionally want to keep older generated images.
shutil.rmtree(SYNTHETIC_DCGAN_DATA, ignore_errors=True)
shutil.rmtree(SYNTHETIC_ACGAN_DATA, ignore_errors=True)
SYNTHETIC_DCGAN_DATA.mkdir(parents=True, exist_ok=True)
SYNTHETIC_ACGAN_DATA.mkdir(parents=True, exist_ok=True)

# GAN images must be 256x256 for the repo DCGAN discriminator.
# We convert to greyscale, then repeat to 3 channels because the repo GAN classes expect RGB-like input.
img_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

full_ds = datasets.ImageFolder(str(LABELLED_DATA), transform=img_tf)
CLASSES = full_ds.classes
class_to_idx = full_ds.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}
LOOKUP_LABEL = class_to_idx

print('classes:', CLASSES)
print('class_to_idx:', class_to_idx)

# Sanity check: repo DCGAN discriminator expects 256x256 real images.
_debug_loader = DataLoader(full_ds, batch_size=min(BATCH_SIZE, 4), shuffle=True, num_workers=0)
_debug_x, _debug_y = next(iter(_debug_loader))
print('GAN batch shape:', tuple(_debug_x.shape))
assert _debug_x.shape[1:] == (3, 256, 256), f'Expected [B, 3, 256, 256], got {_debug_x.shape}'
del _debug_loader, _debug_x, _debug_y


def make_class_loader(class_name):
    target_idx = class_to_idx[class_name]
    inds = [i for i, (_, y) in enumerate(full_ds.samples) if y == target_idx]
    if MAX_IMAGES_PER_CLASS is not None:
        inds = inds[:MAX_IMAGES_PER_CLASS]
    ds = Subset(full_ds, inds)
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=True,
    )


bce = nn.BCELoss()
ce = nn.CrossEntropyLoss()


def denorm(x):
    return (x + 1) / 2


def to_greyscale_generated(imgs):
    """
    Convert generated RGB tensor to single-channel greyscale.
    Input may be in [-1, 1]. Output stays in the same range.
    """
    if imgs.dim() == 3:
        imgs = imgs.unsqueeze(0)
    if imgs.shape[1] == 3:
        imgs = imgs.mean(dim=1, keepdim=True)
    return imgs


def add_instance_noise(imgs, epoch):
    if INSTANCE_NOISE_START <= 0:
        return imgs
    progress = min(max((epoch - 1) / max(GAN_EPOCHS - 1, 1), 0.0), 1.0)
    sigma = INSTANCE_NOISE_START * (1.0 - progress)
    if sigma <= 0:
        return imgs
    return (imgs + torch.randn_like(imgs) * sigma).clamp(-1, 1)


def save_preview_grid(G, class_name, epoch, model_name='DCGAN'):
    G.eval()
    with torch.no_grad():
        if model_name == 'DCGAN':
            z = torch.randn(NUM_PREVIEW, NOISE_DIM, device=DEVICE)
            imgs = G(z)
        else:
            label_id = LOOKUP_LABEL[class_name]
            z = torch.randn(NUM_PREVIEW, ACGAN_NOISE_DIM, device=DEVICE)
            labels = torch.full((NUM_PREVIEW,), label_id, dtype=torch.long, device=DEVICE)
            imgs = G(z, labels)

        imgs = to_greyscale_generated(imgs)
        imgs = denorm(imgs).clamp(0, 1)
        grid = make_grid(imgs, nrow=4, normalize=False)

        out_file = PREVIEW_DIR / f'{model_name}_{class_name}_epoch_{epoch:03d}.png'
        save_image(grid, out_file)
        print('Saved preview:', out_file)


def save_samples_dcgan(G, class_name, n=N_SYNTH_PER_CLASS):
    out_dir = SYNTHETIC_DCGAN_DATA / class_name / 'images'
    out_dir.mkdir(parents=True, exist_ok=True)
    G.eval()
    done = 0
    with torch.no_grad():
        while done < n:
            bs = min(BATCH_SIZE, n - done)
            z = torch.randn(bs, NOISE_DIM, device=DEVICE)
            imgs = G(z)
            imgs = to_greyscale_generated(imgs)
            imgs = denorm(imgs).clamp(0, 1)
            for i in range(bs):
                save_image(imgs[i], out_dir / f'dcgan_{class_name}_{done+i:05d}.png')
            done += bs


def save_samples_acgan(G, class_name, n=N_SYNTH_PER_CLASS):
    out_dir = SYNTHETIC_ACGAN_DATA / class_name / 'images'
    out_dir.mkdir(parents=True, exist_ok=True)
    G.eval()
    label_id = LOOKUP_LABEL[class_name]
    done = 0
    with torch.no_grad():
        while done < n:
            bs = min(BATCH_SIZE, n - done)
            z = torch.randn(bs, ACGAN_NOISE_DIM, device=DEVICE)
            labels = torch.full((bs,), label_id, dtype=torch.long, device=DEVICE)
            imgs = G(z, labels)
            imgs = to_greyscale_generated(imgs)
            imgs = denorm(imgs).clamp(0, 1)
            for i in range(bs):
                save_image(imgs[i], out_dir / f'acgan_{class_name}_{done+i:05d}.png')
            done += bs


histories = []

# ----------------------------
# Train 4 DCGANs, one per class
# ----------------------------
for class_name in CLASSES:
    print('\n==============================')
    print('Training DCGAN for', class_name)
    print('==============================')

    loader = make_class_loader(class_name)
    G = DCGANGenerator(NOISE_DIM, checkpoint_path=str(GAN_OUT)).to(DEVICE)
    D = DCGANDiscriminator().to(DEVICE)

    opt_g = torch.optim.Adam(G.parameters(), lr=LR_G, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(D.parameters(), lr=LR_D, betas=(0.5, 0.999))

    fixed_z = torch.randn(NUM_PREVIEW, NOISE_DIM, device=DEVICE)

    for epoch in range(1, GAN_EPOCHS + 1):
        G.train()
        D.train()
        g_losses, d_losses = [], []

        for batch_idx, (x, _) in enumerate(tqdm(loader, desc=f'DCGAN {class_name} epoch {epoch}/{GAN_EPOCHS}', leave=False)):
            x = x.to(DEVICE)
            bs = x.size(0)

            real_targets = torch.full((bs, 1), REAL_LABEL, device=DEVICE)
            fake_targets = torch.full((bs, 1), FAKE_LABEL, device=DEVICE)

            # ---------------------
            # Train discriminator every 3 batches
            # ---------------------
            if batch_idx % D_TRAIN_EVERY == 0:
                z = torch.randn(bs, NOISE_DIM, device=DEVICE)
                fake_imgs = G(z).detach()

                real_input = add_instance_noise(x, epoch)
                fake_input = add_instance_noise(fake_imgs, epoch)

                d_loss = bce(D(real_input), real_targets) + bce(D(fake_input), fake_targets)

                opt_d.zero_grad()
                d_loss.backward()
                opt_d.step()

                d_losses.append(float(d_loss.item()))

            # ---------------------
            # Train generator every batch
            # ---------------------
            z = torch.randn(bs, NOISE_DIM, device=DEVICE)
            gen_imgs = G(z)
            g_loss = bce(D(gen_imgs), real_targets)

            opt_g.zero_grad()
            g_loss.backward()
            opt_g.step()

            g_losses.append(float(g_loss.item()))

        row = {
            'model': 'DCGAN',
            'class': class_name,
            'epoch': epoch,
            'g_loss': np.mean(g_losses),
            'd_loss': np.mean(d_losses) if d_losses else np.nan,
            'lr_g': LR_G,
            'lr_d': LR_D,
            'd_train_every': D_TRAIN_EVERY,
            'real_label': REAL_LABEL,
            'fake_label': FAKE_LABEL,
            'instance_noise_start': INSTANCE_NOISE_START,
        }
        histories.append(row)
        print(row)

        if epoch == 1 or epoch % PREVIEW_EVERY == 0 or epoch == GAN_EPOCHS:
            # Save fixed-noise preview for easier visual comparison.
            G.eval()
            with torch.no_grad():
                imgs = G(fixed_z)
                imgs = to_greyscale_generated(imgs)
                imgs = denorm(imgs).clamp(0, 1)
                grid = make_grid(imgs, nrow=4, normalize=False)
                out_file = PREVIEW_DIR / f'DCGAN_{class_name}_epoch_{epoch:03d}.png'
                save_image(grid, out_file)
                print('Saved preview:', out_file)

    torch.save(G.state_dict(), GAN_OUT / f'DCGAN_Generator_{class_name}.pth')
    torch.save(D.state_dict(), GAN_OUT / f'DCGAN_Discriminator_{class_name}.pth')
    save_samples_dcgan(G, class_name)

# ----------------------------
# Train 1 ACGAN for all classes
# ----------------------------
print('\n==============================')
print('Training one ACGAN for all classes')
print('==============================')

acgan_loader = DataLoader(
    full_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True,
)

G = ACGANGenerator(
    n_classes=len(CLASSES),
    latent_dim=ACGAN_NOISE_DIM,
    img_size=IMG_SIZE,
    lookup_label=LOOKUP_LABEL,
    out_channels=3,
).to(DEVICE)

D = ACGANDiscriminator(
    n_classes=len(CLASSES),
    img_size=IMG_SIZE,
    out_channels=3,
).to(DEVICE)

opt_g = torch.optim.Adam(G.parameters(), lr=LR_G, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(D.parameters(), lr=LR_D, betas=(0.5, 0.999))

for epoch in range(1, GAN_EPOCHS + 1):
    G.train()
    D.train()
    g_losses, d_losses = [], []

    for batch_idx, (x, y) in enumerate(tqdm(acgan_loader, desc=f'ACGAN epoch {epoch}/{GAN_EPOCHS}', leave=False)):
        x, y = x.to(DEVICE), y.to(DEVICE)
        bs = x.size(0)

        real_targets = torch.full((bs,), REAL_LABEL, device=DEVICE)
        fake_targets = torch.full((bs,), FAKE_LABEL, device=DEVICE)

        # ---------------------
        # Train discriminator every 3 batches
        # ---------------------
        if batch_idx % D_TRAIN_EVERY == 0:
            z = torch.randn(bs, ACGAN_NOISE_DIM, device=DEVICE)
            gen_labels = torch.randint(0, len(CLASSES), (bs,), device=DEVICE)
            fake_imgs = G(z, gen_labels).detach()

            real_input = add_instance_noise(x, epoch)
            fake_input = add_instance_noise(fake_imgs, epoch)

            real_validity, real_cls = D(real_input)
            fake_validity, fake_cls = D(fake_input)

            d_loss = (
                bce(real_validity, real_targets)
                + bce(fake_validity, fake_targets)
                + ce(real_cls, y)
                + ce(fake_cls, gen_labels)
            )

            opt_d.zero_grad()
            d_loss.backward()
            opt_d.step()

            d_losses.append(float(d_loss.item()))

        # ---------------------
        # Train generator every batch
        # ---------------------
        z = torch.randn(bs, ACGAN_NOISE_DIM, device=DEVICE)
        gen_labels = torch.randint(0, len(CLASSES), (bs,), device=DEVICE)
        gen_imgs = G(z, gen_labels)
        validity, pred_cls = D(gen_imgs)

        g_loss = bce(validity, real_targets) + ce(pred_cls, gen_labels)

        opt_g.zero_grad()
        g_loss.backward()
        opt_g.step()

        g_losses.append(float(g_loss.item()))

    row = {
        'model': 'ACGAN',
        'class': 'all',
        'epoch': epoch,
        'g_loss': np.mean(g_losses),
        'd_loss': np.mean(d_losses) if d_losses else np.nan,
        'lr_g': LR_G,
        'lr_d': LR_D,
        'd_train_every': D_TRAIN_EVERY,
        'real_label': REAL_LABEL,
        'fake_label': FAKE_LABEL,
        'instance_noise_start': INSTANCE_NOISE_START,
    }
    histories.append(row)
    print(row)

    if epoch == 1 or epoch % PREVIEW_EVERY == 0 or epoch == GAN_EPOCHS:
        # Save one preview for each class.
        for class_name in CLASSES:
            save_preview_grid(G, class_name, epoch, model_name='ACGAN')

torch.save(G.state_dict(), GAN_OUT / 'ACGAN_Generator.pth')
torch.save(D.state_dict(), GAN_OUT / 'ACGAN_Discriminator.pth')

for class_name in CLASSES:
    save_samples_acgan(G, class_name)

hist_df = pd.DataFrame(histories)
hist_df.to_csv(GAN_OUT / 'gan_training_history.csv', index=False)

print('Saved history:', GAN_OUT / 'gan_training_history.csv')
print('DCGAN synthetic image count:', count_images(SYNTHETIC_DCGAN_DATA))
print('ACGAN synthetic image count:', count_images(SYNTHETIC_ACGAN_DATA))

# Basic proxy comparison. The report used IS/FID; this cell records generation counts and final losses.
summary = hist_df.groupby('model').tail(1).copy()
summary.to_csv(GAN_OUT / 'gan_comparison_summary.csv', index=False)
display(summary)
